# 52nd Place Solution Notebook

This notebook is a cleaned version of my final submission.  
See [this post](https://www.kaggle.com/competitions/h-and-m-personalized-fashion-recommendations/discussion/324076/) for some details about my solution.

The notebook has minimal code in it - most of the code is imported from my [handmhelpers dataset](https://www.kaggle.com/datasets/jacob34/handmhelpers), which is synced to [this github repo](https://github.com/JacobCP/kaggle-handm-helpers) .  
See [this post](https://www.kaggle.com/competitions/h-and-m-personalized-fashion-recommendations/discussion/324078) for some details about my code development.  

**Please note:**  
I plan on continuing to update the github repo, as I try to recreate some of the strategies shared by winning teams.  
Some of those changes may break the code usage for this notebook.  
In order to keep this notebook functional, I will no longer be updating the dataset to reflect the changes made to the repo - it will remain at commit 86c412e902a7692b24e15791322a8dfeb5a761eb

In [1]:
%%time
import os
import sys
import copy
from datetime import datetime
import gc
import pickle as pkl
import shelve

import pandas as pd
import numpy as np
import cudf
    
# Works both on Kaggle (helpers dataset in ../input/) and locally (git submodule at repo root)
REPO_ROOT = os.path.abspath("..")
sys.path.append(REPO_ROOT if os.path.isdir(os.path.join(REPO_ROOT, "handmhelpers")) else "../input/")

KAGGLE_DATA_DIR = "/kaggle/input/h-and-m-personalized-fashion-recommendations/"
DATA_DIR = KAGGLE_DATA_DIR if os.path.isdir(KAGGLE_DATA_DIR) else os.path.join(REPO_ROOT, "data")
OUTPUT_DIR = "." if os.path.isdir(KAGGLE_DATA_DIR) else os.path.join(REPO_ROOT, "outputs")
from handmhelpers import io as h_io, sub as h_sub, cv as h_cv, fe as h_fe
from handmhelpers import modeling as h_modeling, candidates as h_can, pairs as h_pairs

CPU times: user 1.85 s, sys: 364 ms, total: 2.22 s
Wall time: 4.65 s


## Load and convert data

In [2]:
%%time

c, t, a = h_io.load_data(data_path=DATA_DIR, files=['customers.csv', 'transactions_train.csv', 'articles.csv'])        

index_to_id_dict_path = h_fe.reduce_customer_id_memory(c, [t])
t["week_number"] = h_fe.day_week_numbers(t["t_dat"])
t["t_dat"] = h_fe.day_numbers(t["t_dat"])

CPU times: user 5.47 s, sys: 2.65 s, total: 8.12 s
Wall time: 46.9 s


# Get item pairs

In [3]:
%%time

pairs_per_item = 5

week_number_pairs = {}
for week_number in [96, 97, 98, 99, 100, 101, 102, 103, 104]:
    print(f"Creating pairs for week number {week_number}")
    week_number_pairs[week_number] = h_pairs.create_pairs(
        t, week_number, pairs_per_item, verbose=False
    )

Creating pairs for week number 96
Creating pairs for week number 97
Creating pairs for week number 98
Creating pairs for week number 99
Creating pairs for week number 100
Creating pairs for week number 101
Creating pairs for week number 102
Creating pairs for week number 103
Creating pairs for week number 104
CPU times: user 41.8 s, sys: 24.8 s, total: 1min 6s
Wall time: 1min 7s


## Main retrieval/features function!

In [4]:
def create_candidates_with_features_df(t, c, a, customer_batch=None, **kwargs):
    # splitting cv
    features_df, label_df = h_cv.feature_label_split(
        t, kwargs["label_week"], kwargs["feature_periods"]
    )
    
    # converting relative day_number
    features_df["t_dat"] = h_fe.how_many_ago(features_df["t_dat"])
    features_df["week_number"] = h_fe.how_many_ago(features_df["week_number"])
    
    # pull out the cv week
    article_pairs_df = week_number_pairs[kwargs["label_week"]-1]
    
    # check if we can limit customers
    if len(label_df) > 0:
        customers = label_df["customer_id"].unique()
    elif customer_batch is not None:
        customers = customer_batch
    else:
        customers = None
    
    ############################################
    # creating candidates (and adding features)
    ###########################################
    
    features_db = shelve.open("features_db") 
    
    # creating candidate (and saving features created)
    recent_customer_cand, features_db["customer_article"] = (
        h_can.create_recent_customer_candidates(
            features_df,
            kwargs["ca_num_weeks"],
            customers=customers,
        )
    )
    
    (cust_last_week_cand,
     cust_last_week_pair_cand,
     features_db["clw"],
     features_db["clw_pairs"]) = h_can.create_last_customer_weeks_and_pairs(
        features_df,
        article_pairs_df,
        kwargs["clw_num_weeks"],
        kwargs["clw_num_pair_weeks"],
        customers=customers,
    )
    
    _, features_db["popular_articles"] = h_can.create_popular_article_cand(
        features_df,
        c,
        a,
        kwargs["pa_num_weeks"],
        kwargs["hier_col"],
        num_candidates=kwargs["num_recent_candidates"],
        num_articles=kwargs["num_recent_articles"],
        customers=customers,
    )
    age_bucket_can, _, _ = h_can.create_age_bucket_candidates(
        features_df,
        c,
        kwargs["num_age_buckets"],
        articles=kwargs["num_recent_articles"],
        customers=customers,
    )
    
    cand = [recent_customer_cand, cust_last_week_cand, cust_last_week_pair_cand, age_bucket_can]
    cand = cudf.concat(cand).drop_duplicates()
    cand = cand.sort_values(["customer_id", "article_id"]).reset_index(drop=True)
    
    del recent_customer_cand, cust_last_week_cand, cust_last_week_pair_cand, age_bucket_can
    
    cand = h_can.filter_candidates(cand, t, **kwargs)
    
    # creating other features
    h_fe.create_cust_hier_features(features_df, a, kwargs["hier_cols"], features_db)
    h_fe.create_price_features(features_df, features_db)
    h_fe.create_cust_features(c, features_db)
    h_fe.create_article_cust_features(features_df, c, features_db)
    h_fe.create_lag_features(features_df, a, kwargs["lag_days"], features_db)
    h_fe.create_rebuy_features(features_df, features_db)
    h_fe.create_cust_t_features(features_df, a, features_db)
    h_fe.create_art_t_features(features_df, features_db)
    
    del features_df

    # another filter at the end, for the ones that didn't get filtered earlier
    if customers is not None:
        cand = cand[cand["customer_id"].isin(customers)]
    
    # report on recall/precision of candidates
    if kwargs["cv"]:
        ground_truth_candidates = label_df[["customer_id", "article_id"]].drop_duplicates()
        h_cv.report_candidates(cand, ground_truth_candidates)
        del ground_truth_candidates        
    
    # adding features to candidates
    cand_with_f_df = h_can.add_features_to_candidates(
        cand, features_db, c, a
    )
    
    # manually adding article features (couldn't use shelve for some reason)
    for article_col in kwargs["article_columns"]:
        art_col_map = a.set_index("article_id")[article_col]
        cand_with_f_df[article_col] = cand_with_f_df["article_id"].map(art_col_map)
    
    # limiting features
    if kwargs["selected_features"] is not None:
        cand_with_f_df = cand_with_f_df[
            ["customer_id", "article_id"] + kwargs["selected_features"]
        ]
        
    features_db.close()
    os.remove("features_db.bak"), os.remove("features_db.dir"), os.remove("features_db.dat")
    
    assert len(cand) == len(cand_with_f_df), "seem to have duplicates in the feature dfs"
    del cand
    
    return cand_with_f_df, label_df

In [5]:
def calculate_model_score(ids_df, preds, truth_df):
    predictions = h_modeling.create_predictions(ids_df, preds)
    true_labels = h_cv.ground_truth(truth_df).set_index("customer_id")["prediction"]
    score = round(h_cv.comp_average_precision(true_labels, predictions),5)
    
    return score

## Parameters - one place for all!

In [6]:
cv_params = {
    "cv": True,
    "feature_periods": 105,
    "label_week": 104,
    "index_to_id_dict_path": index_to_id_dict_path,
    "pairs_file_version": "_v3_5_ex",
    "num_recent_candidates": 36,
    "num_recent_articles": 12,
    "hier_col": "department_no",
    "ca_num_weeks": 3,
    "clw_num_weeks": 12,
    "clw_num_pair_weeks": 2,
    "pa_num_weeks": 1,
    "num_age_buckets": 4,
    "filter_recent_art_weeks": 1,
    "filter_num_articles": None,
    "lag_days": [1, 3, 14, 30],
    "article_columns": ["index_code"],
    "hier_cols": [
        "department_no", "section_no", "index_group_no", "index_code",
        "product_type_no", "product_group_name"
    ],
    "selected_features": None,
    "lgbm_params": {"n_estimators": 200, "num_leaves": 20},
    "log_evaluation": 10,
    "early_stopping": 20,
    "eval_at": 12,
    "save_model": True,
    "num_concats": 5,
}
sub_params = {
    "cv": False,
    "feature_periods": 105,
    "label_week": 105,
    "index_to_id_dict_path": index_to_id_dict_path,
    "pairs_file_version": "_v3_5_ex",
    "num_recent_candidates": 60,
    "num_recent_articles": 12,
    "hier_col": "department_no",
    "ca_num_weeks": 3,
    "clw_num_weeks": 12,
    "clw_num_pair_weeks": 2,
    "pa_num_weeks": 1,
    "num_age_buckets": 4,
    "filter_recent_art_weeks": 1,
    "filter_num_articles": None,
    "lag_days": [1, 3, 14, 30],
    "article_columns": ["index_code"],
    "hier_cols": [
        "department_no", "section_no", "index_group_no", "index_code",
        "product_type_no", "product_group_name"
    ],
    "selected_features": None,
    "lgbm_params": {
        "n_estimators": 150,
        "num_leaves": 20,    
    },
    "log_evaluation": 10,
    "eval_at": 12,
    "prediction_models": ["model_104", "model_105"],
    "save_model": True,
    "num_concats": 5,
}

In [7]:
cand_features_func = create_candidates_with_features_df
scoring_func = calculate_model_score

In [8]:
%%time
cv_weeks = [104]
results = h_modeling.run_all_cvs(
    t, c, a, cand_features_func, scoring_func, 
    cv_weeks=cv_weeks, **cv_params
)

preparing training modeling dfs for 103...
candidates recall: 7.37% (16,790/227,910)
candidates precision: 0.73% (16,790/2,294,052)
preparing training modeling dfs for 102...
candidates recall: 7.47% (17,783/238,074)
candidates precision: 0.75% (17,783/2,360,656)
preparing training modeling dfs for 101...
candidates recall: 7.03% (17,937/255,172)
candidates precision: 0.71% (17,937/2,541,984)
preparing training modeling dfs for 100...
candidates recall: 7.15% (16,493/230,825)
candidates precision: 0.70% (16,493/2,355,379)
preparing training modeling dfs for 99...
candidates recall: 7.14% (16,923/237,160)
candidates precision: 0.68% (16,923/2,473,752)
concatenating all weeks together
preparing evaluation modeling dfs...
candidates recall: 7.98% (17,062/213,728)
candidates precision: 0.82% (17,062/2,090,136)


/opt/conda/lib/python3.7/site-packages/lightgbm/basic.py:1780: UserWarning: Overriding the parameters from Reference Dataset.
  _log_warning('Overriding the parameters from Reference Dataset.')
/opt/conda/lib/python3.7/site-packages/lightgbm/basic.py:1513: UserWarning: categorical_column in param dict is overridden.
  _log_warning(f'{cat_alias} in param dict is overridden.')


Training until validation scores don't improve for 20 rounds
[10]	train's map@12: 0.30694	train's ndcg@12: 0.405042	validation's map@12: 0.296407	validation's ndcg@12: 0.395729
[20]	train's map@12: 0.315011	train's ndcg@12: 0.414146	validation's map@12: 0.304471	validation's ndcg@12: 0.404279
[30]	train's map@12: 0.320312	train's ndcg@12: 0.420051	validation's map@12: 0.306204	validation's ndcg@12: 0.406651
[40]	train's map@12: 0.32337	train's ndcg@12: 0.423487	validation's map@12: 0.306543	validation's ndcg@12: 0.407588
[50]	train's map@12: 0.325904	train's ndcg@12: 0.426286	validation's map@12: 0.310081	validation's ndcg@12: 0.411151
[60]	train's map@12: 0.328499	train's ndcg@12: 0.428819	validation's map@12: 0.311203	validation's ndcg@12: 0.412928
[70]	train's map@12: 0.331077	train's ndcg@12: 0.431453	validation's map@12: 0.312824	validation's ndcg@12: 0.41385
[80]	train's map@12: 0.334038	train's ndcg@12: 0.434349	validation's map@12: 0.313555	validation's ndcg@12: 0.414788
[90]	t

In [9]:
%%time
gc.collect()
h_modeling.full_sub_train_run(t, c, a, cand_features_func, scoring_func, **sub_params)
predictions = h_modeling.full_sub_predict_run(
    t, c, a, cand_features_func, **sub_params
)

preparing training modeling dfs for 104...
preparing training modeling dfs for 103...
preparing training modeling dfs for 102...
preparing training modeling dfs for 101...
preparing training modeling dfs for 100...
concatenating all weeks together
[1]	train's map@12: 0.256871	train's ndcg@12: 0.355112
[2]	train's map@12: 0.28327	train's ndcg@12: 0.382335
[3]	train's map@12: 0.291793	train's ndcg@12: 0.390663
[4]	train's map@12: 0.294484	train's ndcg@12: 0.393172
[5]	train's map@12: 0.297284	train's ndcg@12: 0.395933
[6]	train's map@12: 0.298776	train's ndcg@12: 0.397691
[7]	train's map@12: 0.301966	train's ndcg@12: 0.401157
[8]	train's map@12: 0.3045	train's ndcg@12: 0.403826
[9]	train's map@12: 0.305557	train's ndcg@12: 0.40508
[10]	train's map@12: 0.307635	train's ndcg@12: 0.407487
[11]	train's map@12: 0.308756	train's ndcg@12: 0.408644
[12]	train's map@12: 0.309535	train's ndcg@12: 0.409755
[13]	train's map@12: 0.310727	train's ndcg@12: 0.410769
[14]	train's map@12: 0.311714	train's

In [10]:
sub = h_sub.create_sub(c["customer_id"], predictions, index_to_id_dict_path)
sub.to_csv(os.path.join(OUTPUT_DIR, 'submission.csv'), index=False)

display(sub.head())
print(sub.shape)

,customer_id,prediction
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,0779781015 0568601043 0762846031 0858856005 08...
1,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,0924243002 0915529005 0919273002 0924243001 08...
2,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,0794321007 0794321008 0918522001 0805000001 09...
3,00005ca1c9ed5f5146b52ac8639a40ca9d57aeff4d1bd2...,0861803009 0852584001 0730683062 0918522001 07...
4,00006413d8573cd20ed7128e53b7b13819fe5cfc2d801f...,0791587021 0730683050 0896152001 0791587001 09...


(1371980, 2)
